# Working Fluid Property Exploration

Comparing thermodynamic properties of multiple working fluids (Water, R-134a, CO2) across a range of conditions using CoolProp.

In [ ]:
import CoolProp.CoolProp as CP
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
fluids = {
    "Water": {"T_range": (280, 640), "color": "#2980b9"},
    "R134a": {"T_range": (250, 370), "color": "#e74c3c"},
    "CO2": {"T_range": (220, 300), "color": "#27ae60"},
}

records = []
for fluid, props in fluids.items():
    T_min, T_max = props["T_range"]
    T_crit = CP.PropsSI("Tcrit", fluid)
    T_range = np.linspace(T_min, min(T_max, T_crit - 2), 100)
    for T in T_range:
        try:
            records.append({
                "Fluid": fluid,
                "T [K]": T,
                "T [C]": T - 273.15,
                "P_sat [kPa]": CP.PropsSI("P", "T", T, "Q", 0, fluid) / 1e3,
                "h_fg [kJ/kg]": (
                    CP.PropsSI("H", "T", T, "Q", 1, fluid)
                    - CP.PropsSI("H", "T", T, "Q", 0, fluid)
                ) / 1e3,
                "s_f [kJ/(kg·K)]": CP.PropsSI("S", "T", T, "Q", 0, fluid) / 1e3,
                "s_g [kJ/(kg·K)]": CP.PropsSI("S", "T", T, "Q", 1, fluid) / 1e3,
            })
        except Exception:
            continue
df = pd.DataFrame(records)
df.head(10)

In [ ]:
fig = px.line(
    df, x="T [C]", y="P_sat [kPa]", color="Fluid",
    title="Saturation Pressure vs. Temperature",
)
fig.update_layout(width=800, height=500)
fig.show()

In [ ]:
fig = px.line(
    df, x="T [C]", y="h_fg [kJ/kg]", color="Fluid",
    title="Latent Heat of Vaporization vs. Temperature",
)
fig.update_layout(width=800, height=500)
fig.show()

In [ ]:
critical_data = []
for fluid in fluids:
    critical_data.append({
        "Fluid": fluid,
        "T_crit [K]": round(CP.PropsSI("Tcrit", fluid), 2),
        "T_crit [C]": round(CP.PropsSI("Tcrit", fluid) - 273.15, 2),
        "P_crit [MPa]": round(CP.PropsSI("Pcrit", fluid) / 1e6, 3),
        "rho_crit [kg/m3]": round(CP.PropsSI("rhocrit", fluid), 2),
    })
df_crit = pd.DataFrame(critical_data)
df_crit

In [ ]:
fig = go.Figure()
for fluid, props in fluids.items():
    fluid_df = df[df["Fluid"] == fluid]
    s_f = fluid_df["s_f [kJ/(kg·K)]"].values
    s_g = fluid_df["s_g [kJ/(kg·K)]"].values
    T = fluid_df["T [C]"].values
    fig.add_trace(go.Scatter(
        x=np.concatenate([s_f, s_g[::-1]]),
        y=np.concatenate([T, T[::-1]]),
        fill="toself",
        fillcolor=props["color"],
        opacity=0.15,
        line=dict(color=props["color"], width=2),
        name=fluid,
    ))
fig.update_layout(
    title="T-s Saturation Envelopes — Fluid Comparison",
    xaxis_title="Entropy [kJ/(kg·K)]",
    yaxis_title="Temperature [C]",
    width=900,
    height=600,
)
fig.show()

## Summary

Water has the widest saturation dome and highest latent heat, making it ideal for power generation. R-134a operates at lower temperatures, suitable for refrigeration. CO2's low critical temperature enables transcritical cycle operation.